# Gemma-WorkOrder：一键完整实验

先在 Colab 选择 T4 GPU，并在 Secrets 中配置已开启 Notebook access 的 `HF_TOKEN`。然后点击：**运行时 → 全部运行**。

本实验是 90 条受控自建样本上的 Base-vs-QLoRA 对照，不用于工业诊断、自动派单或维修决策。

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('未读取到 HF_TOKEN：请在 Colab Secrets 中创建它并开启 Notebook access。')
os.environ['HF_TOKEN'] = token
!nvidia-smi

In [ ]:
# 下载/更新项目，安装依赖，并移除会与 PEFT 合并冲突的旧 torchao。
repo = Path('/content/gemma-inference-eval')
if not repo.exists():
    !git clone -b agent/gemma-workorder-roadmap https://github.com/zmh2245749337/gemma-inference-eval.git /content/gemma-inference-eval
else:
    !git -C /content/gemma-inference-eval fetch origin agent/gemma-workorder-roadmap
    !git -C /content/gemma-inference-eval checkout agent/gemma-workorder-roadmap
    !git -C /content/gemma-inference-eval pull origin agent/gemma-workorder-roadmap
%cd /content/gemma-inference-eval
!pip uninstall -y torchao || true
!pip -q install -r requirements-colab.txt

In [ ]:
# 构建数据、Base Gemma 对照评测、QLoRA 微调、Adapter 评测。
!python scripts/build_workorder_dataset.py --output-dir data/workorder --samples 90 --seed 42
!python scripts/evaluate_workorder.py --model-id google/gemma-3-1b-it --precision 4bit --dataset data/workorder/test.jsonl --output reports/workorder_base_4bit.json
!python scripts/train_workorder_qlora.py --model-id google/gemma-3-1b-it --train data/workorder/train.jsonl --validation data/workorder/validation.jsonl --output-dir artifacts/workorder_qlora_adapter --epochs 3 --learning-rate 2e-4 --batch-size 1 --gradient-accumulation 8 --max-length 1024 --seed 42
!python scripts/evaluate_workorder.py --model-id google/gemma-3-1b-it --adapter artifacts/workorder_qlora_adapter --precision 4bit --dataset data/workorder/test.jsonl --output reports/workorder_qlora_4bit.json

In [ ]:
# 合并 Adapter，并将合并后的 checkpoint 接回自研 Decoder 做白盒数值对齐。
!python scripts/merge_workorder_adapter.py --model-id google/gemma-3-1b-it --adapter artifacts/workorder_qlora_adapter --output-dir artifacts/workorder_merged
!python scripts/run_core_alignment.py --model-id artifacts/workorder_merged --precision fp16 --output reports/workorder_merged_core_alignment.json

In [ ]:
# 最终汇总。只将这里的实际运行数字写进简历或复盘。
import json

for label, filename in [('Base Gemma', 'reports/workorder_base_4bit.json'), ('QLoRA Gemma', 'reports/workorder_qlora_4bit.json')]:
    report = json.loads(Path(filename).read_text(encoding='utf-8'))
    print(f'\n{label}')
    for key, value in report['metrics'].items():
        print(f'  {key}: {value}')

alignment = json.loads(Path('reports/workorder_merged_core_alignment.json').read_text(encoding='utf-8'))
print('\n白盒对齐最后 Token top-1 一致：', alignment['last_token_logits']['top1_match'])